# 🔥 Advanced · Whisper (speech-to-text)

> 🔥 **Advanced / heavy lab.** Fine-tune OpenAI's Whisper on a small ASR dataset — audio is the other half of egocentric video.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **30–60 min** including downloads. Built on the official **[transformers / Whisper](https://huggingface.co/blog/fine-tune-whisper)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Egocentric clips carry narration & sound; Whisper transcribes it (track C).

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — whisper-tiny / base | A100 40 GB — whisper-large-v3 |
| **Storage** | MINDS-14 subset ~ 0.5 GB + model | Common Voice / LibriSpeech 10–60 GB; large-v3 ~ 3 GB |
| **Time** | 300 steps ~ 30–60 min | large-v3 full ~ 1–3 days multi-GPU |

**Full pipeline (scale-up):** `accelerate launch run_speech_recognition_seq2seq.py …` on the full corpus.

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip install -q "transformers>=4.41" datasets accelerate evaluate jiwer librosa soundfile

## 1 · A small speech dataset

In [ ]:
from datasets import load_dataset, Audio
ds = load_dataset("PolyAI/minds14", "en-US", split="train[:200]")
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

## 2 · Processor + model

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
ck = "openai/whisper-tiny"
proc = WhisperProcessor.from_pretrained(ck, language="english", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(ck)
def prep(b):
    a = b["audio"]
    b["input_features"] = proc(a["array"], sampling_rate=16000).input_features[0]
    b["labels"] = proc(text=b["transcription"]).input_ids
    return b
ds = ds.map(prep, remove_columns=ds.column_names)

## 3 · Train

In [ ]:
import torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
def collate(feats):
    inp = proc.feature_extractor.pad([{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
    lab = proc.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
    labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
    inp["labels"] = labels; return inp
args = Seq2SeqTrainingArguments(output_dir="whisper-ft", per_device_train_batch_size=8, learning_rate=1e-5,
    max_steps=300, fp16=torch.cuda.is_available(), logging_steps=25, report_to="none")
Seq2SeqTrainer(model=model, args=args, train_dataset=ds, data_collator=collate, processing_class=proc).train()

## Evaluate — Word Error Rate (WER) on a held-out split (lower = better)

In [ ]:
import evaluate, torch
from datasets import load_dataset, Audio
wer = evaluate.load("wer")
model.to("cuda" if torch.cuda.is_available() else "cpu").eval()
ev = load_dataset("PolyAI/minds14", "en-US", split="train[200:260]").cast_column("audio", Audio(sampling_rate=16000))
preds, refs = [], []
for b in ev:
    feat = proc(b["audio"]["array"], sampling_rate=16000, return_tensors="pt").input_features.to(model.device)
    with torch.no_grad(): ids = model.generate(feat, language="en", task="transcribe", max_new_tokens=128)
    preds.append(proc.batch_decode(ids, skip_special_tokens=True)[0]); refs.append(b["transcription"])
print("WER:", round(100 * wer.compute(predictions=preds, references=refs), 1), "%")

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/ropedia-<lab>")`.

## How this links to tracks A–D
Speech transcripts feed **LM** video-language models (egocentric narration, track C).

## Next steps
- Evaluate **WER** with `evaluate.load("wer")` on a held-out split.
- Fine-tune on **egocentric narration** (Ego4D) and feed transcripts to the Video-LM lab.